# Syft format comparison patterns: one package set, four consumer contracts

last_verified: 2026-10-11 · tool_version: n/a · sources: [] (no externally-verifiable claims; every reference below is an in-repo file)

## Purpose

Syft can emit the same Software Bill of Materials (SBOM) in several output formats — native JSON, SPDX JSON, CycloneDX JSON, and the GitHub dependency-snapshot shape. This notebook compares the four shapes from the consumer's side: given an identical package set, what each format keeps, what it drops, and which downstream job each one serves.

How this differs from the sibling notebook `syft/notebooks/output-format-comparison.ipynb`: that notebook generates each format with the Syft binary from a live image pull. This notebook builds small synthetic fixtures in the four shapes and runs entirely offline on the standard library, so the comparison is reproducible from a clean checkout with no network, no registry access, and no cluster. The question here is not generation but selection — which format to request for which pipeline consumer, and what fidelity loss to expect.

## When to use

- Choosing the output format for a pipeline step that feeds a vulnerability scanner, a license audit, GitHub's dependency graph, or local debugging.
- Debugging a consumer that reports a package as missing when the SBOM looks complete — usually a wrong identity key (`version` vs `versionInfo`, `bom-ref` vs `SPDXID`) rather than a real gap.
- Reviewing an SBOM-storage decision: native JSON retains the most per-package detail, while the exchange formats trade detail for consumer compatibility.

## Prerequisites

- Python 3, standard library only (`json`). No Syft binary, no image pull, no cluster.

In [ ]:
# last_verified: 2026-10-11 · syft n/a
import json

# Synthetic fixture packages. These values are illustrative, not scanned output:
# the point is that the SAME three packages appear in all four envelopes below.
PACKAGES = [
    {"name": "openssl", "version": "3.3.0", "type": "apk",
     "license": "Apache-2.0",
     "purl": "pkg:apk/alpine/openssl@3.3.0?arch=x86_64",
     "location": "/lib/apk/db/installed"},
    {"name": "busybox", "version": "1.36.1", "type": "apk",
     "license": "GPL-2.0-only",
     "purl": "pkg:apk/alpine/busybox@1.36.1?arch=x86_64",
     "location": "/lib/apk/db/installed"},
    {"name": "zlib", "version": "1.3.1", "type": "apk",
     "license": "Zlib",
     "purl": "pkg:apk/alpine/zlib@1.3.1?arch=x86_64",
     "location": "/lib/apk/db/installed"},
]

# Envelope shapes mirror the kit's own reference (syft/docs/sbom-output-formats-reference.md):
# native uses `artifacts`, SPDX uses `packages` + `relationships`,
# CycloneDX uses `components` + `dependencies`, GitHub keeps a minimal manifest map.
# Spec-revision keys are deliberately omitted: the fixtures target structure, not a spec revision.
native_sb = {
    "artifacts": [
        {"id": "fixture-%d" % i, "name": p["name"], "version": p["version"],
         "type": p["type"], "locations": [{"path": p["location"]}],
         "licenses": [p["license"]], "purl": p["purl"]}
        for i, p in enumerate(PACKAGES)
    ],
    "source": {"type": "image", "target": {"userInput": "fixture-image"}},
    "descriptor": {"name": "syft"},
}

spdx_sb = {
    "SPDXID": "SPDXRef-DOCUMENT",
    "name": "fixture-image",
    "packages": [
        {"SPDXID": "SPDXRef-fixture-%d" % i, "name": p["name"],
         "versionInfo": p["version"], "filesAnalyzed": False,
         "licenseConcluded": p["license"], "licenseDeclared": p["license"],
         "externalRefs": [{"referenceCategory": "PACKAGE-MANAGER",
                             "referenceType": "purl",
                             "referenceLocator": p["purl"]}]}
        for i, p in enumerate(PACKAGES)
    ],
    "relationships": [
        {"spdxElementId": "SPDXRef-DOCUMENT",
         "relatedSpdxElement": "SPDXRef-fixture-%d" % i,
         "relationshipType": "DESCRIBES"}
        for i in range(len(PACKAGES))
    ],
}

cyclonedx_sb = {
    "bomFormat": "CycloneDX",
    "metadata": {"component": {"name": "fixture-image", "type": "container"}},
    "components": [
        {"bom-ref": p["purl"], "type": "library", "name": p["name"],
         "version": p["version"],
         "licenses": [{"license": {"id": p["license"]}}],
         "purl": p["purl"]}
        for p in PACKAGES
    ],
    "dependencies": [
        {"ref": p["purl"], "dependsOn": []} for p in PACKAGES
    ],
}

github_sb = {
    "version": 1,
    "job": {"correlator": "fixture-job", "id": "fixture"},
    "detector": {"name": "syft"},
    "manifests": {
        "fixture-image": {
            "resolved": {
                p["name"]: {"package_url": p["purl"]} for p in PACKAGES
            }
        }
    },
}

print("fixtures built: native=%d spdx=%d cyclonedx=%d github=%d" % (
    len(native_sb["artifacts"]), len(spdx_sb["packages"]),
    len(cyclonedx_sb["components"]),
    len(github_sb["manifests"]["fixture-image"]["resolved"])))


In [ ]:
# Step 1 — identity parity: every format describes the same package set.
# Each format names the identity fields differently, so extract per-format first.
native_names = {a["name"] for a in native_sb["artifacts"]}
spdx_names = {p["name"] for p in spdx_sb["packages"]}
cdx_names = {c["name"] for c in cyclonedx_sb["components"]}
github_names = set(github_sb["manifests"]["fixture-image"]["resolved"])
expected = {p["name"] for p in PACKAGES}

assert native_names == expected, native_names
assert spdx_names == expected, spdx_names
assert cdx_names == expected, cdx_names
assert github_names == expected, github_names

# PURL survives in all four, but under a different key each time.
spdx_purls = {r["referenceLocator"]
              for p in spdx_sb["packages"] for r in p["externalRefs"]}
cdx_purls = {c["purl"] for c in cyclonedx_sb["components"]}
github_purls = {v["package_url"]
                for v in github_sb["manifests"]["fixture-image"]["resolved"].values()}
native_purls = {a["purl"] for a in native_sb["artifacts"]}
assert spdx_purls == cdx_purls == github_purls == native_purls
print("parity ok: %d packages, identical PURL set in all four formats" % len(expected))


In [ ]:
# Step 2 — fidelity matrix: what each envelope keeps and what it drops.
# "full" = field present per package; "partial" = present but compressed;
# "absent" = the consumer must look elsewhere (or re-scan) for this detail.
FIDELITY = {
    "name + version":        {"native": "full", "spdx": "full", "cyclonedx": "full", "github": "partial"},
    "purl identity":         {"native": "full", "spdx": "full", "cyclonedx": "full", "github": "full"},
    "license":               {"native": "full", "spdx": "full", "cyclonedx": "full", "github": "absent"},
    "file locations":        {"native": "full", "spdx": "absent", "cyclonedx": "absent", "github": "absent"},
    "graph edges":           {"native": "absent", "spdx": "full", "cyclonedx": "full", "github": "absent"},
}
for row, cells in FIDELITY.items():
    print("%-16s native=%-7s spdx=%-7s cyclonedx=%-7s github=%s" % (row, cells["native"], cells["spdx"], cells["cyclonedx"], cells["github"]))

# Structural spot-checks behind the matrix.
assert all("locations" in a for a in native_sb["artifacts"])
assert len(spdx_sb["relationships"]) == len(spdx_sb["packages"])
assert len(cyclonedx_sb["dependencies"]) == len(cyclonedx_sb["components"])
assert "locations" not in json.dumps(github_sb)
# GitHub keeps name -> package_url only; per-package license detail is dropped.
assert all(set(v) == {"package_url"}
            for v in github_sb["manifests"]["fixture-image"]["resolved"].values())
print("fidelity ok: native keeps locations, SPDX/CycloneDX keep graph edges, GitHub is minimal")


In [ ]:
# Step 3 — selection pattern: map each downstream consumer to its format,
# and compare envelope sizes for the same package set.
def pick_format(consumer):
    """Return the Syft -o format for a downstream consumer."""
    mapping = {
        "vuln-scan": "cyclonedx-json",      # scanners correlate on components/bom-ref
        "license-audit": "spdx-json",        # auditors read licenseConcluded + relationships
        "dependency-graph": "github-json",   # dependency graph takes the minimal snapshot
        "debug-pipeline": "syft-json",       # local debugging wants locations + full metadata
    }
    return mapping[consumer]

assert pick_format("vuln-scan") == "cyclonedx-json"
assert pick_format("license-audit") == "spdx-json"
assert pick_format("dependency-graph") == "github-json"
assert pick_format("debug-pipeline") == "syft-json"

sizes = {
    "syft-json": len(json.dumps(native_sb, sort_keys=True)),
    "spdx-json": len(json.dumps(spdx_sb, sort_keys=True)),
    "cyclonedx-json": len(json.dumps(cyclonedx_sb, sort_keys=True)),
    "github-json": len(json.dumps(github_sb, sort_keys=True)),
}
for name in sorted(sizes, key=sizes.get):
    print("%-14s %5d bytes" % (name, sizes[name]))
assert sizes["github-json"] == min(sizes.values())
print("selection ok: minimal envelope is github-json, richest is syft-json")


## What the comparison shows

- **Native JSON (`artifacts`)** is the only envelope that retains file locations per package. It is the format for debugging the pipeline itself or feeding Syft-ecosystem tooling that needs the full field set.
- **SPDX (`packages` + `relationships`)** is the license-and-provenance exchange shape: every package carries `licenseConcluded`/`licenseDeclared` and a stable `SPDXID` that the document-level `relationships` reference. Reach for it when the SBOM leaves the team (auditors, other toolchains).
- **CycloneDX (`components` + `dependencies`)** keys every component on `bom-ref` and is the shape vulnerability-correlation consumers expect. Reach for it when the next step is scanning or registry risk correlation.
- **GitHub snapshot (`manifests` -> `resolved`)** is deliberately minimal — name to `package_url` only — and is the shape to write when the SBOM uploads straight into a dependency graph. Anything needing licenses or locations must read a richer format instead.

The underlying packages are identical in all four; what changes is the envelope, the identity key (`purl` vs `externalRefs[].referenceLocator` vs `bom-ref` vs `package_url`), and how much per-package detail survives.

## Verify

1. Run every cell top to bottom from a clean checkout: all `assert` statements pass with no network and no Syft binary installed.
2. Parity cell confirms the same package-name set and the same PURL set in all four envelopes.
3. Fidelity cell confirms native-only `locations`, SPDX/CycloneDX graph edges (`relationships`/`dependencies`), and the minimal GitHub `resolved` map.

## Common errors

- **Joining formats on the wrong identity key.** The version field is `version` in native and CycloneDX but `versionInfo` in SPDX; the PURL is `purl` in native and CycloneDX, `externalRefs[].referenceLocator` in SPDX, and `package_url` in the GitHub snapshot. Normalize to PURL before diffing across formats.
- **Expecting file locations outside native JSON.** SPDX, CycloneDX, and GitHub shapes drop install paths; if a triage needs the path on disk, re-read the native SBOM rather than assuming the exchange format lost a package.
- **Reading an empty `dependsOn` as a scan failure.** Container-image SBOMs routinely carry empty dependency edges (as in the kit's own reference); it means the image layer carries no recorded dependency graph, not that cataloging failed.

## References

- `syft/docs/sbom-output-formats-reference.md` — field-level structure of the three main formats
- `syft/docs/output-format-selection-guide.md` — format-selection guidance this notebook's selector follows
- `syft/docs/sbom-formats-comparison.md` — prose comparison companion to this executable one
- `syft/docs/syft-format-migration-guide.md` — rolling a pipeline from one format to another
- `syft/notebooks/output-format-comparison.ipynb` — sibling notebook that generates each format with the Syft binary from a live image